# Step 08.5 — What RNN Solves and What It Costs：RNN 阶段收束

Step 8.1–8.4 已经把 RNN 从动机、公式、时间循环一路写到真正的 `nn.Module` Cell。

现在暂时不增加新架构，而是回答两个问题：

```text
RNN 相比普通 MLP 到底真正解决了什么？
这些解决方案又带来了什么新的结构性代价？
```

本节五部分：① RNN 的三项核心收益；② hidden state 作为历史摘要的优势与 bottleneck；③ parameter sharing 与 runtime cost 的区别；④ 为什么时间依赖形成串行 critical path；⑤ 把所有问题整理成 Step 9 的 BPTT / vanishing-exploding gradients / LSTM-GRU 动机。

目标不是宣判“RNN 好/坏”，而是准确理解它为什么曾经重要、又为什么后来必须继续演化。


## 1. RNN 真正解决了什么？——顺序、共享参数、流式状态

### A. 顺序（order）进入了计算图

Vanilla RNN：

$$
h_t=\tanh(W_xx_t+W_hh_{t-1}+b)
$$

由于 `h_t` 依赖 `h_(t-1)`，交换输入顺序通常会改变后续状态。顺序不是靠把 position index 拼进去才出现，而是由 recurrence 的计算路径天然产生。

### B. 同一套参数可以处理不同 T

`W_x / W_h / b` 的 shape 只依赖 C、H，不依赖 sequence length T。因此同一个 Cell 可以连续处理 3、30、300 个时间步；参数量不必随着 T 增加。

### C. 支持真正的 streaming state

如果 token 一个一个到来，RNN 可以保留当前 `h_t`，下一次只需要新 `x_(t+1)` 与旧 state，就能继续：

```text
x_0 -> h_0
       ↓
x_1 -> h_1
       ↓
x_2 -> h_2
       ↓
new x_3 -> h_3
```

这与每次把整个历史 flatten 成固定长度向量是不同的计算范式。

但这些优点都来自同一个设计：**把历史不断压进一个固定 width H 的状态，并让状态沿时间递推。** 后面的所有代价也从这里产生。


In [ ]:
import mlx.core as mx
import mlx.nn as nn

class RNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.x_proj = nn.Linear(input_size, hidden_size, bias=True)
        self.h_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def __call__(self, x_t, h_prev):
        return mx.tanh(self.x_proj(x_t) + self.h_proj(h_prev))

def run_sequence(cell, X, h0=None):
    B, T, _ = X.shape
    h = mx.zeros((B, cell.hidden_size)) if h0 is None else h0
    states = []
    for t in range(T):
        h = cell(X[:, t, :], h)
        states.append(h)
    return mx.stack(states, axis=1), h

C, H = 3, 4
cell = RNNCell(C, H)

X_short = mx.ones((2, 3, C))
X_long  = mx.ones((2, 7, C))

H_short, final_short = run_sequence(cell, X_short)
H_long, final_long = run_sequence(cell, X_long)
mx.eval(H_short, H_long, final_short, final_long)

print('same cell, T=3 ->', H_short.shape, final_short.shape)
print('same cell, T=7 ->', H_long.shape, final_long.shape)


这个实验只证明**结构和参数可以复用到不同 T**。它不表示随机初始化的 RNN 已经学会任何语言规律，也不表示任意长 T 都能同样好地保留历史信息。


## 2. Hidden State：它既是“记忆”，也是信息瓶颈

RNN 的核心优势是：过去不用全部原样保留，而是被压缩到：

```text
h_t.shape = (B,H)
```

从 streaming 角度看，这非常高效：下一步只需要当前 state，而不一定需要重新读取所有旧输入。

但这里必须准确理解“memory”这个词。

`h_t` 不是数据库，也不是完整历史的无损拷贝。它只是一个由当前参数和更新规则产生的**压缩表示（compressed representation）**。

假设 H=2，那么无论已经读过 5 个 token 还是 500 个 token，下一时刻传递下去的 recurrent state 仍然只有两个 feature coordinates。

这不严格意味着数学上任何长序列都必然无法被编码——连续实数在理想无限精度下可以有复杂编码能力。真正工程中使用有限精度、有限 H、平滑可训练映射和噪声/优化限制，因此长期历史压进固定 state 会成为实际 bottleneck。

用一个故意简单的 scalar recurrence 看“历史细节可能消失”：

$$
h_t=x_t+h_{t-1}.
$$

`[1,2]` 与 `[2,1]` 最终都得到 3。这个例子不是说所有 RNN 都不区分顺序；它只是展示：**压缩状态的具体更新规则完全可能让不同历史汇聚到相同状态。**


In [ ]:
def additive_state(sequence):
    h = 0.0
    states = []
    for x in sequence:
        h = h + x
        states.append(h)
    return states

seq_a = [1.0, 2.0]
seq_b = [2.0, 1.0]

states_a = additive_state(seq_a)
states_b = additive_state(seq_b)

print('sequence A states:', states_a)
print('sequence B states:', states_b)
print('different histories, same final state:', states_a[-1] == states_b[-1])


真正的训练型 RNN 使用 learned `W_x/W_h` 和 tanh，当然可以学到更有用、更顺序敏感的压缩方式。但它依然必须解决一个问题：

> 哪些旧信息应该留下？哪些可以忘掉？

Vanilla RNN 没有显式 gate 告诉状态“保留多少、忘掉多少”。这正是后面 LSTM / GRU 会重新设计的地方。


## 3. 参数量不随 T 增长 ≠ 计算成本不随 T 增长

这是 RNN 中非常重要的工程区别。

一个 Vanilla RNN Cell 的参数量：

$$
P=HC+H^2+H.
$$

这里没有 T，所以 sequence 变长不会创建新 `W_x/W_h/b`。

但每个时间步都要做一次：

```text
x_t @ W_x.T      ~ B*C*H
h_prev @ W_h.T   ~ B*H*H
tanh / bias      ~ B*H
```

忽略常数后，完整长度 T 的 forward compute 大致随：

$$
O\left(TB(CH+H^2)\right)
$$

线性增长。

所以：

```text
parameter count     -> 对 T 基本不变
number of RNN steps -> 与 T 成正比
forward compute     -> 与 T 成正比（固定 B,C,H）
```

此外，如果只做 streaming inference 且只关心当前/final state，持久 recurrent state 可以只有 `(B,H)`；如果要输出全部 states，则至少要保存 `(B,T,H)`。训练 BPTT 时通常还需要保留更多中间计算，实际内存也会随着 T 增长，具体大小取决于框架、checkpointing 等实现。


In [ ]:
from mlx.utils import tree_flatten
import matplotlib.pyplot as plt

C, H = 16, 32
cell = RNNCell(C, H)
parameter_count = sum(v.size for _, v in tree_flatten(cell.parameters()))

lengths = [1, 2, 4, 8, 16, 32, 64]
parameter_counts = [parameter_count for _ in lengths]
recurrent_steps = lengths

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(lengths, parameter_counts, marker='o', label='parameter count')
ax.set_xlabel('sequence length T')
ax.set_ylabel('number of trainable parameters')
ax.set_title('RNN parameters stay constant as T changes')
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

print('parameter count:', parameter_count)
print('recurrent step counts for these T values:', recurrent_steps)


这张图只说明**参数量**不随 T 增长。不要从水平线误读成“运行时间也不变”。

如果要研究速度，我们需要真正 benchmark，不应该仅凭参数量图推断 wall-clock time。


## 4. 最大结构代价：Time Axis 形成 Sequential Critical Path

RNN 的依赖链是：

```text
h_0
 ↓
h_1
 ↓
h_2
 ↓
...
 ↓
h_(T-1)
```

即使 GPU 能一次并行计算 Batch 中很多 samples，`h_t` 仍需要来自前一个时间步的真实 `h_(t-1)`。

所以 sequence length T 不只是“多了 T 份独立工作”，而是形成长度约 T 的 **critical path（关键依赖路径）**。

8.3 已经看到，`X @ W_x.T` 这条 input projection 可以提前对全部 T 一起算；但 recurrent history path 仍然必须等待 previous state。

这对硬件非常重要：GPU 擅长大量规则矩阵运算并行，而长时间递推会在时间轴上限制可并行度。

我们可以用依赖图而不是速度数字来表达这件事。


In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.set_xlim(-0.5, 7.5)
ax.set_ylim(-0.5, 1.5)
ax.axis('off')

T_visual = 7
for t in range(T_visual):
    ax.text(t, 0.5, f'h{t}', ha='center', va='center',
            bbox={'boxstyle':'round,pad=0.3', 'fill':False})
    if t > 0:
        ax.annotate('', xy=(t-0.35, 0.5), xytext=(t-1+0.35, 0.5),
                    arrowprops={'arrowstyle':'->'})

ax.set_title('RNN recurrent critical path grows with sequence length T')
plt.show()


图上的箭头表示数学依赖，不等同于实际每个 kernel 的调度时间。

这也是为什么以后 Transformer 去掉 recurrence 后，一个重要收益是：在训练阶段，各位置的主要 projection/attention 计算能够获得更强的并行性。具体 Self-Attention 的并行结构等到 Step 11 再正式推导。


## 5. 为什么下一步必须学习 BPTT？——从 Forward Memory 到 Credit Assignment

到现在我们只回答了 forward：

```text
x_0 -> h_0 -> h_1 -> ... -> h_(T-1)
```

真正训练时还要回答更困难的问题：

> 如果最后的 prediction 错了，那么很早以前的 `W_h` / `W_x` 应该怎样修改？

由于同一套参数在每个 t 重复使用，loss 对参数的影响会沿着整个展开后的 recurrent graph 反向传播。

概念上：

```text
forward:
x0 -> h0 -> h1 -> h2 -> loss

backward:
loss -> h2 -> h1 -> h0 -> shared parameters
```

这就是 **Backpropagation Through Time (BPTT)**。

而 RNN 的两个核心困难也会在 BPTT 中被放大：

### 问题 1：长距离 credit assignment

很早的状态影响要穿过很多 recurrent transitions 才到达后面的 loss。训练必须判断“几十步之前哪个输入/参数导致今天的错误”。

### 问题 2：重复的导数/Jacobian 连乘

因为同一种 recurrent transformation 被重复很多次，反向传播会出现许多局部导数的乘积。如果这些量长期小于 1，影响可能衰减；如果长期大于 1，可能放大。

下一节 Step 9.1 会真正把 RNN 沿时间展开并用 chain rule 推出这个结构；Step 9.2 再用 `0.5^n`、`1.5^n` 等最小数字例子解释 vanishing / exploding gradients。

随后：

```text
Vanilla RNN
↓ long-range credit assignment is difficult
LSTM
↓ gated memory path
GRU
↓ simplified gated recurrence
```

所以 LSTM/GRU 不是凭空出现的新 API，而是对 Vanilla RNN 长期记忆和训练困难的直接回应。

### Step 8 最终总结

- RNN 用 recurrent hidden state 把 order 与 history 直接写入计算图。
- 时间参数共享让参数量不依赖 T，同一个 cell 可以处理不同 sequence lengths。
- RNN 很适合流式更新：新 token 到来时可从已有 state 继续。
- hidden state 是压缩历史，不是无限、无损记忆；固定 H 会成为实际信息 bottleneck。
- 参数量与 T 无关，但 forward compute、输出 states 和训练图成本仍随 T 增长。
- Batch 可以并行；Time 有 `h_(t-1)->h_t` 的 sequential critical path。
- RNN 的最大训练难题不是“能不能 forward”，而是如何把远处 loss 的责任分配回早期时间步。
- **Step 8 到此完成。下一步 Step 9.1：Backpropagation Through Time。**
